# Real-Time Vessel Tracking with Python and WebSocket Streaming

This notebook is a companion to the Developer Community article [Real-Time Vessel Tracking with Python and WebSocket Streaming](https://developers.lseg.com/en/article-catalog/article/vessel-tracking-part-2-vessel-tracking-websocket-streaming).

### What We'll Build

We'll build a Jupyter notebook that:

- authenticates with the streaming service and connects to it
- subscribes to a vessel and keeps the connection alive
- receives position updates as they are reported
- shows the vessel's latest position, and the path it has followed since you started, on a world map and on a zoomed-in close-up map

### Prerequisites

Before getting started, make sure you have the following:

- Python 3.10 or later
- A Python environment with the Jupyter Notebook extension (VS Code) or JupyterLab, and these libraries installed:
  - websockets==16.1.1
  - ipyleaflet==0.20.0
  - ipywidgets==8.1.9
- A Vessel Tracking API key that is entitled to the vessel you want to watch

This tutorial assumes basic familiarity with Python and Jupyter notebooks.

If you don't already have a Vessel Tracking API key, contact your LSEG account representative or refer to the Vessel Tracking API documentation for instructions on generating one from the Vessel Tracking web application.

Store the **full** authorization value in a `config.ini` file next to the notebook, and keep that file out of version control:

```ini
[vessel_tracking]
api_key = apikey-v1 <your key>
```

We use ipyleaflet to display the maps. The maps use OpenStreetMap tiles, so the notebook needs internet access for them.

### 1. Import the libraries

Uncomment the `%pip` line once to install the libraries, and restart the kernel if asked. Everything else comes from Python's standard library.

In [1]:
# %pip install websockets==16.1.1 ipyleaflet==0.20.0 ipywidgets==8.1.9

import asyncio
import configparser
import contextlib
import html
import json
import urllib.error
import urllib.parse
import urllib.request
from datetime import datetime, timezone

from IPython.display import display
from ipyleaflet import CircleMarker, Map, Polyline, basemaps
from ipywidgets import HTML, Layout, VBox
from websockets.asyncio.client import connect
from websockets.exceptions import ConnectionClosed

### 2. Configure the notebook

Set the service host, the vessel to watch (by IMO number) and the API key. The host is the one used in the REST examples of the Vessel Tracking API documentation; use the host of your own deployment if it differs. The default IMO is a vessel from the documentation's examples, so replace it with a vessel you are entitled to see.

A position is shown in gray when it is older than `STALE_SECONDS`, so you can tell a vessel that has gone quiet from one that is simply moving slowly.

In [2]:
HOST = 'vtracking.rdms.refinitiv.com'
IMO = '9765835'
ORIGIN = 'python-jupyter'
KEEPALIVE_SECONDS = 120
STALE_SECONDS = 1800

config = configparser.ConfigParser()
config.read('config.ini')
API_KEY = config['vessel_tracking']['api_key']

### 3. Get a ticket, build the messages and read the latest position

These small functions map directly to the steps above:

- `get_ticket()` calls the `/auth` endpoint (step 1).
- `connect_url()` builds the `wss://` address from the ticket (step 2).
- `subscribe_message()`, `keepalive_message()` and `unsubscribe_message()` build the JSON messages (steps 3 to 5).
- `get_latest_position()` asks the REST API for the vessel's latest position. REST field names start with a capital letter, so we rename them to match the lower camel case names used by the stream.

The ticket URL contains a temporary credential, so never print it.

In [3]:
def get_ticket():
    request = urllib.request.Request(f'https://{HOST}/VesselTrackerWebSockets/auth',
                                     headers={'Authorization': API_KEY, 'Accept': 'application/json'})
    with urllib.request.urlopen(request, timeout=20) as response:
        return json.load(response)

def connect_url(ticket):
    query = urllib.parse.urlencode({key: ticket[key] for key in ('user', 'label', 'timestamp')})
    return f'wss://{HOST}/VesselTrackerWebSockets/connect?{query}'

def subscribe_message(label):
    return {'method': 'subscribe', 'origin': ORIGIN, 'streamID': label, 'vessels': [{'type': 'imo', 'value': IMO}]}

def keepalive_message(label):
    return {'method': 'keepalive', 'streamID': label}

def unsubscribe_message(label):
    return {'method': 'unsubscribe', 'origin': ORIGIN, 'streamID': label, 'vessels': []}

FIELDS = {'IMO': 'imo', 'VesselName': 'vesselName', 'Latitude': 'latitude', 'Longitude': 'longitude',
          'TimestampPosition': 'timestampPosition', 'SpeedOverGround': 'speedOverGround',
          'AISDestination': 'aisDestination', 'NavigationalStatus': 'navigationalStatus'}

def get_latest_position():
    request = urllib.request.Request(f'https://{HOST}/api/v1/Shipping/Latest?IMOs={IMO}',
                                     headers={'Authorization': API_KEY, 'Accept': 'application/json'})
    with urllib.request.urlopen(request, timeout=20) as response:
        row = json.load(response)[0]
    values = dict(zip(row['fieldNames'], row['fieldValues']))
    return {FIELDS[name]: value for name, value in values.items() if name in FIELDS}

### 4. Keep the latest position and the path

The notebook uses these fields from the position data. The stream spells field names in lower camel case (`latitude`), while the REST field list starts with a capital letter (`Latitude`). Not every field is guaranteed in every stream message, so `update_position()` keeps the values it already has.

| Field | Description |
| --- | --- |
| `latitude` / `longitude` | Position of the vessel |
| `timestampPosition` | When the position was reported (UTC) |
| `imo` / `vesselName` | Vessel identity |
| `speedOverGround` | Reported speed in knots |
| `aisDestination` | Destination reported through AIS |
| `navigationalStatus` | Operational status of the vessel |

`position` holds the vessel's latest position and `trail` holds every distinct location received so far, which we will draw as a line.

`update_position()` accepts a position from either the REST snapshot or the stream. It ignores updates that are older than the one we already have, or that have no usable coordinates, and adds new locations to the trail. Timestamps without a time zone are treated as UTC.

`handle_message()` decodes one message from the stream and ignores anything that is not valid JSON. The server may send a list of positions or a single one, so we handle both. If the message is an `expiryWarning` we reply with a keepalive immediately; every message that carries coordinates is passed to `update_position()`.

In [4]:
position = {}
trail = []
status = {'state': 'Disconnected', 'updates': 0, 'last_message': 'none yet'}

def parse_time(text):
    parsed = datetime.fromisoformat(text.replace('Z', '+00:00'))
    return parsed if parsed.tzinfo else parsed.replace(tzinfo=timezone.utc)

def update_position(record):
    try:
        latitude, longitude = float(record['latitude']), float(record['longitude'])
        time = parse_time(record['timestampPosition'])
    except (KeyError, TypeError, ValueError):
        return
    if position and time < position['_time']:
        return
    position.update(record, latitude=latitude, longitude=longitude, _time=time)
    if not trail or trail[-1] != (latitude, longitude):
        trail.append((latitude, longitude))
    status['updates'] += 1

async def handle_message(raw, socket, label):
    status['last_message'] = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    try:
        payload = json.loads(raw)
    except ValueError:
        return
    for record in payload if isinstance(payload, list) else [payload]:
        if record.get('method') == 'expiryWarning':
            await socket.send(json.dumps(keepalive_message(label)))
        elif 'latitude' in record:
            update_position(record)

### 5. Prepare the world map and the close-up map

This cell only creates the maps; section 7 displays them. A vessel moves only a fraction of a degree per hour, so one map cannot show both where it is in the world and how it is moving. We use two:

- The **world map** shows the whole world, so you see where the vessel is.
- The **close-up map** is zoomed in and recenters on the vessel at every refresh, so small changes in position are visible. Because it recenters, it cannot be panned while tracking runs.

On both maps a blue line shows where the vessel has been, small gray dots mark its earlier positions, and the large dot at the end is the latest position. That dot is red, or gray when the position is older than `STALE_SECONDS`. Click the dot on the world map to see the vessel's details.

`draw()` creates the line and dots for one map the first time and updates them afterwards. `render_map()` runs once per second, so the notebook updates the maps in place instead of creating a new figure for each message.

In [5]:
world_map = Map(center=(15, 10), zoom=1.5, zoom_snap=0.25, min_zoom=1, world_copy_jump=True, basemap=basemaps.OpenStreetMap.Mapnik,
                layout=Layout(height='520px', width='100%'), scroll_wheel_zoom=True)
close_map = Map(center=(15, 10), zoom=7, basemap=basemaps.OpenStreetMap.Mapnik,
                layout=Layout(height='360px', width='100%'), scroll_wheel_zoom=True)
status_widget = HTML(value='Disconnected')
popup_widget = HTML()
world_layers = {}
close_layers = {}

def draw(target, layers, color, radius):
    location = (position['latitude'], position['longitude'])
    if not layers:
        layers['line'] = Polyline(locations=[], color='#2563eb', weight=3)
        layers['dot'] = CircleMarker(location=location, radius=radius, color=color, fill_color=color, fill_opacity=0.9)
        layers['earlier'] = []
        target.add(layers['line'])
        target.add(layers['dot'])
    layers['line'].locations = list(trail)
    for point in trail[len(layers['earlier']):-1]:
        layers['earlier'].append(CircleMarker(location=point, radius=3, color='#6b7280', fill_color='#9ca3af', fill_opacity=1))
        target.add(layers['earlier'][-1])
    layers['dot'].location = location
    layers['dot'].color = color
    layers['dot'].fill_color = color
    return layers['dot']

def render_map():
    text = f"{status['state']} | Positions received: {status['updates']} | Last message received: {status['last_message']}"
    if position:
        age_seconds = (datetime.now(timezone.utc) - position['_time']).total_seconds()
        color = '#6b7280' if age_seconds > STALE_SECONDS else '#dc2626'
        popup_widget.value = '<br>'.join(html.escape(str(line)) for line in [
            position.get('vesselName', IMO), f"Position time: {position['timestampPosition']} UTC",
            f"Speed: {position.get('speedOverGround', 'n/a')} knots", f"Destination: {position.get('aisDestination', 'n/a')}",
            f"Status: {position.get('navigationalStatus', 'n/a')}"])
        draw(world_map, world_layers, color, 7).popup = popup_widget
        draw(close_map, close_layers, color, 9)
        close_map.center = (position['latitude'], position['longitude'])
        text += f" | {position.get('vesselName', IMO)}: latest position is {int(age_seconds // 60)} min old"
    status_widget.value = html.escape(text)

### 6. Stream the positions

`stream()` follows the steps from the introduction in a loop:

1. Read the latest position from REST, so the map has something to show.
2. Get a **new ticket**, because a ticket works only once.
3. Connect and send the subscribe message.
4. Start `send_keepalives()` in the background, then handle every message that arrives.
5. When the loop ends, for example because you pressed stop, send `unsubscribe` and close the connection.

If the connection drops, `stream()` waits five seconds and starts again from step 1. Errors such as a rejected API key stop the stream and are shown in the status line. A production application would also add a growing delay between retries and monitoring.

`run_tracking()` runs the stream and the map refresh together, and `start_tracking()` and `stop_tracking()` start and cancel that task.

In [6]:
async def send_keepalives(socket, label):
    while True:
        await asyncio.sleep(KEEPALIVE_SECONDS)
        await socket.send(json.dumps(keepalive_message(label)))

async def stream():
    while True:
        try:
            status['state'] = 'Reading the latest position'
            update_position(await asyncio.to_thread(get_latest_position))
            status['state'] = 'Getting a ticket'
            ticket = await asyncio.to_thread(get_ticket)
            label = ticket['label']
            async with connect(connect_url(ticket), additional_headers={'Authorization': API_KEY}) as socket:
                await socket.send(json.dumps(subscribe_message(label)))
                status['state'] = 'Subscribed, waiting for updates'
                keepalive = asyncio.create_task(send_keepalives(socket, label))
                try:
                    async for raw in socket:
                        await handle_message(raw, socket, label)
                finally:
                    keepalive.cancel()
                    with contextlib.suppress(ConnectionClosed):
                        await socket.send(json.dumps(unsubscribe_message(label)))
        except urllib.error.HTTPError:
            raise
        except (ConnectionClosed, OSError):
            status['state'] = 'Connection lost, retrying in 5 seconds'
            await asyncio.sleep(5)

async def refresh_map():
    while True:
        render_map()
        await asyncio.sleep(1)

async def run_tracking():
    streaming = asyncio.create_task(stream())
    refreshing = asyncio.create_task(refresh_map())
    try:
        await asyncio.gather(streaming, refreshing)
    except asyncio.CancelledError:
        status['state'] = 'Stopped'
        raise
    except Exception as error:
        status['state'] = f'Stopped: {error}'
    finally:
        streaming.cancel()
        refreshing.cancel()
        await asyncio.gather(streaming, refreshing, return_exceptions=True)
        render_map()

tracking_task = None

def start_tracking():
    global tracking_task
    if tracking_task is None or tracking_task.done():
        tracking_task = asyncio.create_task(run_tracking())

def stop_tracking():
    if tracking_task is not None and not tracking_task.done():
        tracking_task.cancel()

### 7. Start tracking and show the maps

Run the next cell. It displays the status line, the world map and the close-up map, then returns immediately while positions stream in. The status line shows how many positions have arrived and the time of the last message from the server, so you can see that the stream is alive even while the vessel has not moved. The vessel appears as soon as the REST snapshot loads, and each streamed update extends its path. The first streamed update can take a couple of minutes, and a slow vessel moves only slightly between updates. Do not rerun the setup cells while tracking is running.

In [7]:
display(VBox([status_widget, world_map, HTML('<b>Close-up</b>'), close_map]))
start_tracking()

### 8. Stop tracking

Run this cell when you want to stop. It cancels the stream, which sends `unsubscribe` and closes the WebSocket. The maps keep the path that has been drawn. Run section 7 to start again.

In [ ]:
stop_tracking()
if tracking_task is not None:
    with contextlib.suppress(asyncio.CancelledError):
        await tracking_task

### 9. Reset the maps

Run this cell after stopping to remove the path and start from a clean map. Saving a notebook does not keep the WebSocket connection open. Before sharing a notebook, clear any outputs that contain licensed vessel data.

In [ ]:
if tracking_task is not None and not tracking_task.done():
    raise RuntimeError('Stop tracking (section 8) before resetting.')
for target, layers in ((world_map, world_layers), (close_map, close_layers)):
    if layers:
        for layer in [layers['line'], layers['dot'], *layers['earlier']]:
            target.remove(layer)
        layers.clear()
position.clear()
trail.clear()
status['updates'] = 0
status['last_message'] = 'none yet'
render_map()

## Troubleshooting

| Symptom | What to check |
|---|---|
| `KeyError: 'vessel_tracking'` | `config.ini` is next to the notebook and has a `[vessel_tracking]` section with `api_key`. Start the kernel from the notebook's folder. |
| Status shows `Stopped: HTTP Error 401` or `403` | The full authorization value (`apikey-v1 ...`), its validity and your entitlement. It is not an OAuth bearer token. |
| `Stopped: HTTP Error 404` or a rejected handshake | The host and the exact paths for your deployment. |
| Connected but nothing moves | Wait: the first update can take a couple of minutes. Also check that you are entitled to the vessel and that it has reported a new position recently. |
| Dot is gray | The latest position is older than `STALE_SECONDS`. The status line shows its age. |
| Connection drops after a few minutes | The keepalive message must be sent every 120 seconds. WebSocket ping and pong frames do not replace it. |
| Blank map | Widget support in your notebook editor, the installed packages, and access to the OpenStreetMap tile servers. |

## Conclusion

In this article, we demonstrated how the Vessel Tracking API's WebSocket interface delivers vessel positions as they are reported. Starting from an API key, we:

- requested a single-use ticket and connected to the streaming service
- subscribed to a vessel and kept the connection alive
- decoded the position messages and kept the latest position and path
- showed the vessel live on a world map and a close-up map

Together with Part 1, which replayed recent history, you can now both look back at where a vessel has been and watch where it is going as new positions are reported.

### References

- Vessel Tracking API documentation
- [*Visualizing Historical Vessel Positions on a World Map with the Vessel Tracking API and Python*](https://developers.lseg.com/en/article-catalog/article/vessel-tracking-part-1-visualizing-historical-vessel-positions) (Part 1)
- [websockets asyncio client documentation](https://websockets.readthedocs.io/en/stable/reference/asyncio/client.html)
- [ipyleaflet documentation](https://ipyleaflet.readthedocs.io/)